<h2>Description</h2>

Dans ce code, nous allons établir un modèle afin de prédire le débit horaire sur les Champs Élysées.

Imports

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.inspection import permutation_importance

doc = 'champs_elysees.csv'

df_final = pd.read_csv('../../datasets_axes_with_all_features/'+ doc, sep=';')

In [2]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9644 entries, 0 to 9643
Columns: 426 entries, Unnamed: 0 to lag_or_35_20
dtypes: float64(411), int64(1), object(14)
memory usage: 31.3+ MB


In [3]:
df_final = df_final.copy()
df_final['Date et heure de comptage'] = pd.to_datetime(df_final['Date et heure de comptage'], errors='coerce')
df_final = df_final.sort_values('Date et heure de comptage').reset_index(drop=True)

for col in ['est_vacances', 'est_ferie', 'est_avant_ferie', 'est_pieton']:
    if col in df_final.columns:
        df_final[col] = pd.to_numeric(df_final[col], errors='coerce')

features = [
    'Température', 'duree prec (en min)', 'precipitations heure',
    'heure_sin', 'heure_cos', 'jour_sin', 'jour_cos', 'mois_sin', 'mois_cos', 
    'force moyenne vent (m/s)', 'jour_semaine', 'est_weekend', 'est_vacances', 'est_avant_vacances',
    'est_ferie', 'est_avant_ferie', 'ensoleillement (en min)', 'est_pieton',
    'lag_dh_6_20', 'lag_or_6_20', 'lag_dh_7_0', 'lag_or_7_0', 'lag_dh_7_4', 'lag_or_7_4', 'lag_dh_7_8', 'lag_or_7_8', 'lag_dh_7_12', 'lag_or_7_12', 'lag_dh_7_16', 'lag_or_7_16', 'lag_dh_7_20', 'lag_or_7_20', 'lag_dh_13_20', 'lag_or_13_20','lag_dh_14_0', 'lag_or_14_0', 'lag_dh_14_4', 'lag_or_14_4', 'lag_dh_14_8', 'lag_or_14_8', 'lag_dh_14_12', 'lag_or_14_12', 'lag_dh_14_16', 'lag_or_14_16', 'lag_dh_14_20', 'lag_or_14_20', 'lag_dh_20_20', 'lag_or_20_20','lag_dh_21_0', 'lag_or_21_0', 'lag_dh_21_4', 'lag_or_21_4', 'lag_dh_21_8', 'lag_or_21_8', 'lag_dh_21_12', 'lag_or_21_12', 'lag_dh_21_16', 'lag_or_21_16', 'lag_dh_21_20', 'lag_or_21_20', 'lag_dh_27_20', 'lag_or_27_20', 'lag_dh_28_0', 'lag_or_28_0', 'lag_dh_28_4', 'lag_or_28_4', 'lag_dh_28_8', 'lag_or_28_8', 'lag_dh_28_12', 'lag_or_28_12', 'lag_dh_28_16', 'lag_or_28_16', 'lag_dh_28_20', 'lag_or_28_20', 'lag_dh_34_20', 'lag_or_34_20', 'lag_dh_35_0', 'lag_or_35_0', 'lag_dh_35_4', 'lag_or_35_4', 'lag_dh_35_8', 'lag_or_35_8', 'lag_dh_35_12', 'lag_or_35_12', 'lag_dh_35_16', 'lag_or_35_16', 'lag_dh_35_20', 'lag_or_35_20'
]

target = 'Débit horaire'

mask_known   = df_final[target].notna()
mask_missing = df_final[target].isna()

X_known = df_final.loc[mask_known, features].copy()
y_known = df_final.loc[mask_known, target].astype(float)
X_missing = df_final.loc[mask_missing, features].copy()

numeric_features = [
    'Température', 'precipitations heure',
    'heure_sin', 'heure_cos', 'jour_sin', 'jour_cos', 'mois_sin', 'mois_cos',
    'force moyenne vent (m/s)', 'duree prec (en min)', 'est_weekend', 'est_vacances', 'est_avant_vacances',
    'est_ferie', 'est_avant_ferie', 'ensoleillement (en min)', 'est_pieton',
    'lag_dh_6_20', 'lag_or_6_20', 'lag_dh_7_0', 'lag_or_7_0', 'lag_dh_7_4', 'lag_or_7_4', 'lag_dh_7_8', 'lag_or_7_8', 'lag_dh_7_12', 'lag_or_7_12', 'lag_dh_7_16', 'lag_or_7_16', 'lag_dh_7_20', 'lag_or_7_20', 'lag_dh_13_20', 'lag_or_13_20','lag_dh_14_0', 'lag_or_14_0', 'lag_dh_14_4', 'lag_or_14_4', 'lag_dh_14_8', 'lag_or_14_8', 'lag_dh_14_12', 'lag_or_14_12', 'lag_dh_14_16', 'lag_or_14_16', 'lag_dh_14_20', 'lag_or_14_20', 'lag_dh_20_20', 'lag_or_20_20','lag_dh_21_0', 'lag_or_21_0', 'lag_dh_21_4', 'lag_or_21_4', 'lag_dh_21_8', 'lag_or_21_8', 'lag_dh_21_12', 'lag_or_21_12', 'lag_dh_21_16', 'lag_or_21_16', 'lag_dh_21_20', 'lag_or_21_20', 'lag_dh_27_20', 'lag_or_27_20', 'lag_dh_28_0', 'lag_or_28_0', 'lag_dh_28_4', 'lag_or_28_4', 'lag_dh_28_8', 'lag_or_28_8', 'lag_dh_28_12', 'lag_or_28_12', 'lag_dh_28_16', 'lag_or_28_16', 'lag_dh_28_20', 'lag_or_28_20', 'lag_dh_34_20', 'lag_or_34_20', 'lag_dh_35_0', 'lag_or_35_0', 'lag_dh_35_4', 'lag_or_35_4', 'lag_dh_35_8', 'lag_or_35_8', 'lag_dh_35_12', 'lag_or_35_12', 'lag_dh_35_16', 'lag_or_35_16', 'lag_dh_35_20', 'lag_or_35_20'
]

categorical_features = ['jour_semaine']

try:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)  
except TypeError:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse=False)         

preprocess = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[('imputer', SimpleImputer(strategy='median'))]), numeric_features),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ohe', ohe)
        ]), categorical_features),
    ],
    remainder='drop'
)

model = HistGradientBoostingRegressor(
    loss='absolute_error',   
    max_depth=15,
    max_iter=300,
    early_stopping=False,
    random_state=42
)

pipe = Pipeline(steps=[('prep', preprocess), ('model', model)])

# ---------- 3) Split chronologique ----------
X_train, X_test, y_train, y_test = train_test_split(
    X_known, y_known, test_size=0.2, shuffle=False
)

# ---------- 4) Pondérations (férié / veille / piéton) ----------
W_FERIE   = 3.0
W_AVANT   = 1.5
W_PIETON  = 10
POST_SCALE_FERIE = 1.0

def make_weights(X_frame):
    w = np.ones(len(X_frame), dtype=float)
    is_ferie  = X_frame['est_ferie'].fillna(0).astype(int).to_numpy()
    is_avant  = X_frame['est_avant_ferie'].fillna(0).astype(int).to_numpy()
    is_pieton = X_frame['est_pieton'].fillna(0).astype(int).to_numpy()

    w[is_ferie == 1]  = W_FERIE
    w[is_avant == 1]  = np.maximum(w[is_avant == 1], W_AVANT)
    w[is_pieton == 1] = W_PIETON

    # Option : normalisation pour garder une échelle de perte comparable
    #w *= (len(w) / w.sum())
    return w

w_train = make_weights(X_train)

# ---------- 5) Entraînement ----------
pipe.fit(X_train, y_train, model__sample_weight=w_train)

# ---------- 6) Prédiction + post-ajustement éventuel ----------
y_pred = pipe.predict(X_test)

mask_ferie_test = X_test['est_ferie'].fillna(0).astype(int).to_numpy() == 1
mask_est_pieton_test = X_test['est_pieton'].fillna(0).astype(int).to_numpy() == 1
y_pred[mask_ferie_test] *= POST_SCALE_FERIE
#y_pred[mask_est_pieton_test] *= 0.5

# ---------- 7) Évaluation ----------
r2   = r2_score(y_test, y_pred)
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R² : {r2:.3f}")
print(f"MAE : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")

# Diagnostics par sous-régimes
is_pieton_test = X_test['est_pieton'].fillna(0).astype(int) == 1
print(f"Part d'observations piéton (test) : {is_pieton_test.mean():.1%}")
if is_pieton_test.any():
    mae_pieton = mean_absolute_error(y_test[is_pieton_test], y_pred[is_pieton_test])
    print(f"MAE (jours piéton) : {mae_pieton:.2f} (n={is_pieton_test.sum()})")
    mae_non_pieton = mean_absolute_error(y_test[~is_pieton_test], y_pred[~is_pieton_test])
    print(f"MAE (jours non piéton) : {mae_non_pieton:.2f} (n={(~is_pieton_test).sum()})")

df_final['Débit_prédit'] = np.nan
df_final.loc[X_test.index, 'Débit_prédit'] = y_pred


R² : 0.770
MAE : 86.34
RMSE : 121.21
Part d'observations piéton (test) : 2.1%
MAE (jours piéton) : 222.83 (n=34)
MAE (jours non piéton) : 83.45 (n=1606)


In [4]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9644 entries, 0 to 9643
Columns: 427 entries, Unnamed: 0 to Débit_prédit
dtypes: datetime64[ns, UTC](1), float64(412), int64(1), object(13)
memory usage: 31.4+ MB


In [13]:
from sklearn.inspection import permutation_importance
import pandas as pd
import numpy as np
import plotly.express as px

# 1) Importance par permutation sur le pipeline complet (prétraitements inclus)
perm = permutation_importance(
    estimator=pipe,
    X=X_test,
    y=y_test,
    n_repeats=20,
    random_state=42,
    scoring='neg_root_mean_squared_error'  # cohérent avec votre RMSE
)

imp_df = (
    pd.DataFrame({
        'feature': features,
        'importance_mean': perm.importances_mean,
        'importance_std': perm.importances_std
    })
    .sort_values('importance_mean', ascending=False)
)

print(imp_df.head(20))

# 2) Bar chart Plotly (top 20)
topk = imp_df.head(20).sort_values('importance_mean', ascending=True)
fig = px.bar(
    topk,
    x='importance_mean', y='feature',
    error_x='importance_std',
    orientation='h',
    title='Importance par permutation — Top 20 (plus haut = plus influent)'
)
fig.update_layout(xaxis_title="Perte de performance (Δ RMSE, signe inversé)", yaxis_title="")
fig.show()


                     feature  importance_mean  importance_std
3                  heure_sin        20.132780        1.466975
20                lag_dh_7_0        19.919265        1.757821
17                est_pieton        11.197515        1.020842
62               lag_dh_28_0         8.060548        1.549175
48               lag_dh_21_0         3.794517        0.979942
4                  heure_cos         2.671885        0.306869
9   force moyenne vent (m/s)         1.308433        0.199301
79               lag_or_35_4         1.194025        0.214848
6                   jour_cos         1.017756        0.191003
5                   jour_sin         0.621908        0.264512
14                 est_ferie         0.584504        0.570440
35               lag_or_14_0         0.563445        0.271861
34               lag_dh_14_0         0.560969        0.545477
10              jour_semaine         0.545094        0.175240
57              lag_or_21_16         0.389419        0.138420
63      

In [6]:
time_index = df_final.loc[X_test.index, 'Date et heure de comptage']

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=time_index,
    y=y_pred,
    mode='lines',
    name='Débit prédit'
))
fig.add_trace(go.Scatter(
    x=time_index,
    y=y_test,
    mode='lines',
    name='Débit réel'
))

fig.update_layout(
    title="Comparaison des débits (réel vs prédit)",
    xaxis_title="Date et heure",
    yaxis_title="Débit horaire (véh/h)",
    hovermode='x unified'
)

fig.show()

In [7]:
X_all = df_final.loc[:, features].copy()

predictions_all = pipe.predict(X_all)

serie = pd.Series(predictions_all, index=df_final.index)

commun = df_final[target].notna() & serie.notna()

predictions_with_know = serie.loc[commun].astype(float)

mae = mean_absolute_error(predictions_with_know, y_known)
rmse = np.sqrt(mean_squared_error(predictions_with_know, y_known))
r2   = r2_score(predictions_with_know, y_known)

print("Nombre de valeurs : " + str(len(predictions_with_know)))
print(f"R² : {r2:.3f}")
print(f"MAE : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")


Nombre de valeurs : 8197
R² : 0.815
MAE : 63.56
RMSE : 110.66


In [8]:


fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_final['Date et heure de comptage'],
    y=predictions_all,
    mode='lines',
    name='Débit prédit'
))
fig.add_trace(go.Scatter(
    x=df_final['Date et heure de comptage'],
    y=df_final['Débit horaire'],
    mode='lines',
    name='Débit réel'
))

fig.update_layout(
    title="Comparaison des débits (réel vs prédit)",
    xaxis_title="Date et heure",
    yaxis_title="Débit horaire (véh/h)",
    hovermode='x unified'
)

fig.show()

<h2>Taux d'occupation</h2>

In [9]:
target_occ = 'Taux d\'occupation'

features_occ = [
    'Température', 'duree prec (en min)', 'Débit_prédit',
    'heure_sin', 'heure_cos', 'mois_sin', 'mois_cos',
    'force moyenne vent (m/s)', 'mois', 'jour_sin',
    'est_ferie', 'est_avant_ferie', 'ensoleillement (en min)', 'est_pieton'
]

mask_occ = df_final[target_occ].notna() & df_final['Débit_prédit'].notna()
X_occ = df_final.loc[mask_occ, features_occ].copy()
y_occ = df_final.loc[mask_occ, target_occ].astype(float)

X_train_occ, X_test_occ, y_train_occ, y_test_occ = train_test_split(
    X_occ, y_occ, test_size=0.2, shuffle=False
)

numeric_features_occ = [c for c in features_occ if c != 'jour_semaine']
categorical_features_occ = []

try:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse=False)

preprocess_occ = ColumnTransformer(
    transformers=[
        ('num', SimpleImputer(strategy='median'), numeric_features_occ),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ohe', ohe)
        ]), categorical_features_occ),
    ],
    remainder='drop'
)

model_occ = HistGradientBoostingRegressor(
    loss='poisson',
    max_depth=5,
    max_iter=40,
    early_stopping=False,
    random_state=42
)

pipe_occ = Pipeline(steps=[('prep', preprocess_occ), ('model', model_occ)])

pipe_occ.fit(X_train_occ, y_train_occ)

y_pred_occ = pipe_occ.predict(X_test_occ)

print("=== Performances taux d'occupation ===")
print(f"R²   : {r2_score(y_test_occ, y_pred_occ):.3f}")
print(f"MAE  : {mean_absolute_error(y_test_occ, y_pred_occ):.2f}")
print(f"RMSE : {np.sqrt(mean_squared_error(y_test_occ, y_pred_occ)):.2f}")
print(f"nb test : {len(y_test_occ)}")


=== Performances taux d'occupation ===
R²   : 0.807
MAE  : 2.82
RMSE : 3.64
nb test : 328


In [10]:
perm = permutation_importance(
    estimator=pipe_occ,
    X=X_test_occ,
    y=y_test_occ,
    n_repeats=20,
    random_state=42,
    scoring='neg_root_mean_squared_error'  
)

imp_df = (
    pd.DataFrame({
        'feature': features_occ,
        'importance_mean': perm.importances_mean,
        'importance_std': perm.importances_std
    })
    .sort_values('importance_mean', ascending=False)
)

print(imp_df.head(20))

topk = imp_df.head(20).sort_values('importance_mean', ascending=True)
fig = px.bar(
    topk,
    x='importance_mean', y='feature',
    error_x='importance_std',
    orientation='h',
    title='Importance par permutation — Top 20 (plus haut = plus influent)'
)
fig.update_layout(xaxis_title="Perte de performance (Δ RMSE, signe inversé)", yaxis_title="")
fig.show()


                     feature  importance_mean  importance_std
2               Débit_prédit         4.332553        0.259306
3                  heure_sin         2.096775        0.162434
4                  heure_cos         0.442351        0.091006
9                   jour_sin         0.342047        0.057530
5                   mois_sin         0.172544        0.047334
12   ensoleillement (en min)         0.156312        0.034092
7   force moyenne vent (m/s)         0.074706        0.072583
1        duree prec (en min)         0.050136        0.021570
0                Température         0.038789        0.051548
6                   mois_cos         0.000000        0.000000
8                       mois         0.000000        0.000000
10                 est_ferie         0.000000        0.000000
11           est_avant_ferie         0.000000        0.000000
13                est_pieton         0.000000        0.000000


In [11]:
time_index_occ = df_final.loc[X_test_occ.index, 'Date et heure de comptage']

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=time_index_occ,
    y=y_pred_occ,
    mode='lines',
    name='Débit prédit'
))
fig.add_trace(go.Scatter(
    x=time_index_occ,
    y=y_test_occ,
    mode='lines',
    name='Débit réel'
))

fig.update_layout(
    title="Comparaison des débits (réel vs prédit)",
    xaxis_title="Date et heure",
    yaxis_title="Débit horaire (véh/h)",
    hovermode='x unified'
)

fig.show()